# End-to-End ML Pipeline with Scikit-learn Pipeline API
This notebook demonstrates how to build a production-ready, reusable machine learning pipeline to predict customer churn using the Telco Churn dataset.

# Objectives:
Handle data preprocessing (imputation, scaling, and encoding) using ColumnTransformer and Pipeline.

Evaluate and tune both LogisticRegression and RandomForestClassifier simultaneously using GridSearchCV.

Export the optimized end-to-end pipeline using joblib for deployment.

In [1]:
# Install required libraries
!pip install -q pandas scikit-learn joblib openpyxl

# 1. Load and Inspect the Dataset
We fetch the dataset directly from IBM's public repository. We'll also perform basic cleanup on the TotalCharges column, which often contains hidden whitespace characters instead of missing values.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the Telco Churn dataset
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Drop CustomerID as it has no predictive power
df.drop(columns=["customerID"], inplace=True)

# Clean TotalCharges: convert spaces to NaN and handle type conversion
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors='coerce')

# Convert target variable 'Churn' to binary numeric values (Yes = 1, No = 0)
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Split features and target
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Split into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training shape: {X_train.shape}")
print(f"Testing shape: {X_test.shape}")

Training shape: (5634, 19)
Testing shape: (1409, 19)


# 2. Build Preprocessing Pipelines
We isolate numeric and categorical columns to
apply different transformations using Scikit-learn's ColumnTransformer.

Numeric Features: Impute missing values with the median, then scale features using StandardScaler.

Categorical Features: Impute missing values with the most frequent value, then encode using OneHotEncoder.

In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define feature types
numeric_features = ["tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [col for col in X.columns if col not in numeric_features]

# Numeric preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing pipeline
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

print("Preprocessing pipeline built successfully.")

Preprocessing pipeline built successfully.


# 3. Define the Complete Pipeline & Hyperparameter Space
We initialize a placeholder model inside our main pipeline. This allows us to use GridSearchCV to search through different algorithms (Logistic Regression vs. Random Forest) and their respective hyperparameters all at once.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# Create an end-to-end pipeline with a placeholder estimator
full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000)) # Placeholder
])

# Define the hyperparameter grid for both models
param_grid = [
    {
        'classifier': [LogisticRegression(max_iter=1000, solver='liblinear')],
        'classifier__C': [0.01, 0.1, 1.0, 10.0],
        'classifier__penalty': ['l1', 'l2']
    },
    {
        'classifier': [RandomForestClassifier(random_state=42)],
        'classifier__n_estimators': [50, 100, 200],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5]
    }
]

# Set up GridSearchCV
grid_search = GridSearchCV(
    estimator=full_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

# Run hyperparameter tuning
grid_search.fit(X_train, y_train)

print(f"\nBest Model Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation F1-Score: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 26 candidates, totalling 130 fits

Best Model Parameters: {'classifier': LogisticRegression(max_iter=1000, solver='liblinear'), 'classifier__C': 10.0, 'classifier__penalty': 'l1'}
Best Cross-Validation F1-Score: 0.5979


# 4. Evaluate Final Performance and Export the Pipeline
We evaluate our optimized production-ready pipeline against the held-out test data and serialize the entire object using joblib.

In [5]:
import joblib
from sklearn.metrics import classification_report, confusion_matrix

# Extract the best model pipeline
best_pipeline = grid_search.best_estimator_

# Generate predictions on test data
y_pred = best_pipeline.predict(X_test)

# Print performance reports
print("--- Final Model Evaluation (Test Set) ---")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Save the complete pipeline (includes preprocessing + trained model)
model_filename = "churn_production_pipeline.joblib"
joblib.dump(best_pipeline, model_filename)
print(f"\nSuccessfully exported pipeline to: {model_filename}")

--- Final Model Evaluation (Test Set) ---
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

Confusion Matrix:
[[926 109]
 [165 209]]

Successfully exported pipeline to: churn_production_pipeline.joblib
